# wrappers.py — User Guide

This notebook walks through every public symbol exported by `financialtools/wrappers.py`.

## Public symbols

| Symbol | Kind | Purpose |
|---|---|---|
| `DownloaderWrapper` | class | Public download API — fetches one or many tickers, merges results, returns a single `pd.DataFrame` |
| `FundamentalEvaluator` | class | Multi-ticker evaluation wrapper around `FundamentalTraderAssistant`; supports parallel execution |
| `export_financial_results` | function | Merges a list of per-ticker `evaluate()` dicts and writes five Excel sheets to `financial_data/` |
| `read_financial_results` | function | Reads those Excel sheets back, optionally filters by ticker and year, returns four DataFrames |
| `merge_results` | function | Concatenates a specific key (e.g. `'composite_scores'`) from a collection of result dicts |

### Logging

`wrappers.py` configures the package logger at **import time** (module level, before any class body).
Three `FileHandler` objects are attached to the `'TickerDownloader'` logger:

- `logs/info.log` — level `INFO`
- `logs/error.log` — level `ERROR`
- `logs/debug.log` — level `DEBUG`

The `logs/` directory is anchored to `os.path.dirname(__file__)/../logs`, meaning it is always
relative to the installed package location, **not** the caller's current working directory.


## Section 2 — Imports


In [ ]:
import pandas as pd
from financialtools.wrappers import (
    DownloaderWrapper,
    FundamentalEvaluator,
    export_financial_results,
    read_financial_results,
    merge_results,
)
from financialtools.processor import _empty_result


## Section 3 — DownloaderWrapper

`DownloaderWrapper` is the public entry point for downloading financial data.

### How it works

- `download_data(tickers)` accepts either a single ticker string or a list of ticker strings.
- Internally it calls `Downloader.from_ticker(ticker)` (from `processor.py`) for each symbol.
  `Downloader.from_ticker` fetches balance-sheet, income-statement, and cash-flow data from
  Yahoo Finance via `yfinance`, then reshapes each table from wide format to long format.
- The per-ticker DataFrames are merged with `get_merged_data()` and pre-processed
  (year extracted from the `time` column; columns lowercased and snake-cased).
- If multiple tickers are requested the results are concatenated into a single DataFrame
  with a `ticker` column identifying the source.
- If **all** downloads fail `download_data` returns `None`; individual failures are skipped
  and the remaining tickers are still processed.
- A 2-second sleep between individual downloads avoids hitting Yahoo Finance rate limits.

### Logging

Every step logs to the three files described in Section 1:

- Start and success of each ticker → `logs/info.log`
- Any exception → `logs/error.log`
- Full traceback on failure → `logs/debug.log`


## Section 4 — DownloaderWrapper demo

> **LIVE** — requires an active internet connection and a writeable `financial_data/` directory.


In [ ]:
# LIVE — requires internet + writeable financial_data/ directory
wrapper = DownloaderWrapper()
df = wrapper.download_data(["AAPL", "MSFT"])
print(f"Shape: {df.shape}")
print(f"Tickers: {df['ticker'].unique()}")
df.head(3)


## Section 5 — FundamentalEvaluator

`FundamentalEvaluator` wraps `FundamentalTraderAssistant` (from `processor.py`) to make
multi-ticker evaluation straightforward.

### Constructor

```python
FundamentalEvaluator(df: pd.DataFrame, weights: pd.DataFrame)
```

- `df` — the merged DataFrame produced by `DownloaderWrapper.download_data()`.  Must contain
  at least the columns `ticker` and `time`.
- `weights` — a sector-weights DataFrame with columns `sector`, `metrics`, and `weights`.
  Build it from `financialtools.config.sector_metric_weights`:
  ```python
  from financialtools.config import sector_metric_weights
  import pandas as pd
  sector = "Technology"
  weights = (
      pd.DataFrame(list(sector_metric_weights[sector].items()), columns=["metrics", "weights"])
      .assign(sector=sector)
  )
  ```

### evaluate_single(ticker) -> dict

- Filters `df` to rows where `ticker == ticker`.
- Constructs a `FundamentalTraderAssistant` and calls `.evaluate()`.
- Returns a dict with five keys: `metrics`, `eval_metrics`, `composite_scores`,
  `raw_red_flags`, `red_flags` — each a `pd.DataFrame`.
- If anything goes wrong (empty slice, `EvaluationError`, any other exception) it logs the
  error and returns `_empty_result()` — a dict of five empty DataFrames — so the caller
  always receives the same shape regardless of success or failure.

### evaluate_multiple(tickers, parallel=True, max_workers=5) -> dict

- Calls `evaluate_single` for every ticker in `tickers`.
- When `parallel=True` (the default) uses `ThreadPoolExecutor(max_workers=max_workers)` to
  run evaluations concurrently.  The default `max_workers=5` limits concurrency to avoid
  triggering yfinance rate limits.  Completion order is non-deterministic.
- When `parallel=False` evaluations run sequentially in list order.
- Returns `{ticker: result_dict}` for every requested ticker.
- A failed ticker receives `_empty_result()` (or `None` for unexpected future errors) and
  does **not** abort the rest of the batch.

## Section 6 — FundamentalEvaluator demo

> **LIVE** — requires data already downloaded by `DownloaderWrapper` and a valid weights DataFrame.


In [ ]:
# LIVE — assumes `df` was produced by DownloaderWrapper.download_data() above.
# Build a weights DataFrame for the target sector from config.
import pandas as pd
from financialtools.config import sector_metric_weights

SECTOR = "Technology"
weights_df = (
    pd.DataFrame(list(sector_metric_weights[SECTOR].items()), columns=["metrics", "weights"])
    .assign(sector=SECTOR)
)

# Construct the evaluator
evaluator = FundamentalEvaluator(df=df, weights=weights_df)  # noqa: F821 — df supplied above

# Evaluate both tickers in parallel (default)
results = evaluator.evaluate_multiple(["AAPL", "MSFT"])

# Inspect one result
aapl_result = results["AAPL"]
print("Result keys:", list(aapl_result.keys()))
print("\nComposite scores for AAPL:")
aapl_result["composite_scores"]

## Section 7 — evaluate_single failure path

When `evaluate_single` cannot find the ticker in `df` (or any other error occurs) it catches
the exception, logs it, and returns `_empty_result()` — a dict of five **empty** DataFrames.
This guarantees the caller always gets the same dict shape and never sees an unhandled exception.


In [ ]:
# Offline: show that a missing ticker returns _empty_result() without raising
import pandas as pd
from financialtools.config import sector_metric_weights

# With an empty dataframe, evaluate_single should return _empty_result safely
empty_df = pd.DataFrame(columns=["ticker", "time"])

SECTOR = "Technology"
weights_df = (
    pd.DataFrame(list(sector_metric_weights[SECTOR].items()), columns=["metrics", "weights"])
    .assign(sector=SECTOR)
)

evaluator = FundamentalEvaluator(df=empty_df, weights=weights_df)
result = evaluator.evaluate_single("AAPL")
print("Result keys:", list(result.keys()))
print("All empty:", all(v.empty for v in result.values()))

## Section 8 — export_financial_results / read_financial_results

These two functions form a persistence round-trip for evaluation results.

### export_financial_results(results, output_dir, sheet_name)

- `results` — a **list** of per-ticker result dicts (each produced by `evaluate()` or
  `evaluate_single()`).
- Internally calls `merge_results(results, key)` for each of the five canonical keys:
  `metrics`, `eval_metrics`, `composite_scores`, `red_flags`, `raw_red_flags`.
- Writes each merged DataFrame to `{output_dir}/{key}.xlsx` using `export_to_xlsx` from
  `financialtools.utils`.
- Creates `output_dir` if it does not exist.
- Default `output_dir` is `"financial_data"`; default `sheet_name` is `"sheet1"`.

### read_financial_results(ticker, time, input_dir, sheet_name)

- Reads back the five Excel files produced by `export_financial_results`.
- Optionally filters each DataFrame by `ticker` and/or `time` (year) if those arguments
  are provided and the corresponding column exists in the sheet.
- All numeric values are rounded to 4 decimal places.
- `red_flags` and `raw_red_flags` sheets are concatenated row-wise before being returned,
  so the caller receives a single combined red-flags DataFrame.
- Returns a 4-tuple: `(metrics, eval_metrics, composite_scores, red_flags)`.
- Any sheet that cannot be read (file missing, wrong sheet name, etc.) silently returns an
  empty DataFrame for that position rather than raising.

### Note on red-flag consolidation

`evaluate()` produces two distinct red-flag DataFrames:

- `raw_red_flags` — cash-flow quality flags (negative FCF, negative OCF, EBITDA vs OCF divergence)
- `red_flags` — metric-threshold flags (negative margins, high D/E, insufficient FCF-to-debt)

`read_financial_results` concatenates both into the single `red_flags` return value.


## Section 9 — merge_results


In [ ]:
# merge_results combines a list of per-ticker evaluate() dicts into wide DataFrames.
# The second argument is the key to extract from each dict.
# Passing a list (not a dict) here — note that merge_results iterates .values(),
# so wrap individual dicts in a list and pass as the positional 'results' argument.

r1 = _empty_result()
r2 = _empty_result()

# merge_results expects a dict {ticker: result_dict}; simulate two tickers
results_dict = {"FAKE1": r1, "FAKE2": r2}
merged = merge_results(results_dict, "metrics")
print(f"Merged metrics shape: {merged.shape}")  # (0, 0) — both empty


## Section 10 — Logging architecture

### Where handlers are registered

`wrappers.py` runs the following at **module import time** (top-level, outside any class or
function):

1. Computes `_LOGS_DIR = os.path.join(os.path.dirname(__file__), '..', 'logs')` — always
   anchored to the package directory, never to the caller's current working directory.
2. Creates `logs/` if it does not exist (`os.makedirs(..., exist_ok=True)`).
3. Acquires the named logger `logging.getLogger('TickerDownloader')` and sets its level to
   `DEBUG`.
4. Attaches three `FileHandler` objects with a shared `%(asctime)s - %(levelname)s - %(message)s`
   formatter:
   - `info.log` at `INFO`
   - `error.log` at `ERROR`
   - `debug.log` at `DEBUG`

### processor.py logging

`processor.py` creates its own logger with `logging.getLogger(__name__)` (i.e. the name
`'financialtools.processor'`).  It does **not** add handlers itself — those are expected to
be provided by the calling environment.  When `wrappers.py` is imported, the handlers
attached to `'TickerDownloader'` service that logger's output.  If you use `processor.py`
standalone (without importing `wrappers.py`), no file handlers are active and log records
are silently discarded unless you configure them yourself.

### Rules to follow

- **Do not** call `logging.basicConfig()` anywhere in application code — it adds a
  `StreamHandler` to the root logger and can cause duplicate console output.
- **Do not** add extra handlers to `'TickerDownloader'` from outside `wrappers.py` — the
  three handlers registered at import time are authoritative.
- If you need a different log directory in tests, patch `_LOGS_DIR` before importing
  `wrappers` or use `importlib.reload` after pointing the path elsewhere.


## Section 11 — End-to-end pattern

The complete pipeline: download → evaluate → persist → read back.

Each stage is wrapped in its own `try/except` so a failure at one stage does not silently
corrupt later stages.


In [ ]:
# LIVE — full pipeline example
# Adjust tickers and sector as needed.

import pandas as pd
from financialtools.wrappers import (
    DownloaderWrapper,
    FundamentalEvaluator,
    export_financial_results,
    read_financial_results,
)
from financialtools.config import sector_metric_weights

TICKERS = ["AAPL", "MSFT"]
SECTOR  = "Technology"

# Build weights from config (no external file required)
weights_df = (
    pd.DataFrame(list(sector_metric_weights[SECTOR].items()), columns=["metrics", "weights"])
    .assign(sector=SECTOR)
)

# --- Stage 1: Download ---
try:
    raw_df = DownloaderWrapper.download_data(TICKERS)
    if raw_df is None:
        raise RuntimeError("All downloads failed — check logs/error.log")
    print(f"Downloaded {raw_df['ticker'].nunique()} ticker(s), {len(raw_df)} rows")
except Exception as exc:
    print(f"Download error: {exc}")
    raise

# --- Stage 2: Evaluate ---
try:
    evaluator = FundamentalEvaluator(df=raw_df, weights=weights_df)
    results = evaluator.evaluate_multiple(TICKERS, parallel=True)
    print(f"Evaluated: {list(results.keys())}")
except Exception as exc:
    print(f"Evaluation error: {exc}")
    raise

# --- Stage 3: Export ---
try:
    export_financial_results(
        results=list(results.values()),
        output_dir="financial_data",
        sheet_name="sheet1"
    )
    print("Exported results to financial_data/*.xlsx")
except Exception as exc:
    print(f"Export error: {exc}")
    raise

# --- Stage 4: Read back ---
try:
    metrics, eval_metrics, composite_scores, red_flags = read_financial_results(
        ticker="AAPL",
        time=None,          # None = all years
        input_dir="financial_data",
        sheet_name="sheet1"
    )
    print(f"metrics rows: {len(metrics)}")
    print(f"composite_scores:\n{composite_scores}")
    print(f"red_flags:\n{red_flags}")
except Exception as exc:
    print(f"Read error: {exc}")
    raise

## Section 12 — Common failure modes

| Symptom | Cause | Fix |
|---|---|---|
| `logs/` directory not created; no log files appear | Importing `wrappers.py` from a location where `os.path.dirname(__file__)` resolves unexpectedly (e.g. a frozen/packaged build) | Verify `_LOGS_DIR` at runtime: `import financialtools.wrappers as w; print(w._LOGS_DIR)` |
| `evaluate_multiple` returns a dict where every value is a dict of empty DataFrames | The `df` passed to `FundamentalEvaluator` contains no rows for those tickers (filter mismatch, wrong `ticker` column casing, or a failed download) | Check `df['ticker'].unique()` before constructing the evaluator; ensure column names are lowercase snake-case (as produced by `DownloaderWrapper`) |
| `read_financial_results` returns empty DataFrames for all four outputs | `export_financial_results` has not been called yet, or it was called with a different `output_dir` | Call `export_financial_results` first and pass the same `output_dir` to both functions; confirm the five `.xlsx` files exist in that directory |
| `FundamentalTraderAssistant` raises `EvaluationError: data contains multiple tickers` | `evaluate_single` passed a slice that still contains more than one ticker | This should not happen through the normal `FundamentalEvaluator` path; if calling `FundamentalTraderAssistant` directly, pre-filter `df` to a single ticker before construction |
| Duplicate log entries (each line appears twice in log files) | `logging.basicConfig()` was called elsewhere, or handlers were added to the root logger independently | Remove any `logging.basicConfig()` calls from application code; check for stray `addHandler` calls on the root or `'TickerDownloader'` logger |
| `merge_results` returns an empty DataFrame even though individual results are non-empty | `merge_results` was called with a **list** instead of a **dict** — it calls `.values()` internally | Pass `{ticker: result_dict, ...}` rather than a bare list of dicts |
